# Chest X-Ray Pneumonia Detection: training run (TensorFlow / Keras)

Trains DenseNet121 on the Kaggle chest X-ray dataset, evaluates it on the 624-image test set, and generates Grad-CAM explanations. Every figure and metric in the repo comes from this run.

**How to run**
1. *Runtime → Change runtime type →* **T4 GPU**
2. *Runtime →* **Run all** (about 30 min)
3. The last cell downloads `results.zip`. Save this notebook with its outputs too: *File → Download → Download .ipynb*

## 1. Environment

In [ ]:
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU found: Runtime -> Change runtime type -> T4 GPU, then run all again."
!nvidia-smi -L

In [ ]:
%cd /content
!rm -rf Chest_Xray_Disease_Detection
!git clone -q https://github.com/SWAPI03/Chest_Xray_Disease_Detection.git
%cd Chest_Xray_Disease_Detection
!git log --oneline -1
!pip install -q -r requirements.txt

## 2. Dataset

Downloads [paultimothymooney/chest-xray-pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) with `kagglehub`. If the download asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's *Secrets* panel (key icon) and rerun this cell.

The Kaggle `val/` folder only has 16 images, so `prepare_data.py` moves 10% of `train/` into `val/` (fixed seed).

In [ ]:
import shutil
from pathlib import Path
import kagglehub

path = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
# The archive holds chest_xray/{train,val,test} plus a duplicate nested copy; use the shallowest.
src = min((p.parent for p in path.rglob("train") if (p / "NORMAL").is_dir()), key=lambda p: len(p.parts))
print("Source:", src)

dst = Path("data/raw")
if not dst.exists():
    for split in ["train", "val", "test"]:
        shutil.copytree(src / split, dst / split)
!python data/prepare_data.py

## 3. A quick look at the data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.dataset import list_split, load_image, CLASSES

splits = {s: list_split("data/raw", s) for s in ["train", "val", "test"]}

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3)
for k, (cls, color) in enumerate(zip(CLASSES, ["#2196F3", "#F44336"])):
    n = [int((splits[s][1] == k).sum()) for s in splits]
    bars = ax.bar(x + (k - 0.5) * 0.38, n, 0.38, label=cls, color=color)
    ax.bar_label(bars, padding=2)
ax.set_xticks(x, [s.capitalize() for s in splits])
ax.set_ylabel("Images")
ax.set_title("Class balance per split", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.legend()
plt.tight_layout()
plt.savefig("outputs/visualizations/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

paths, labels = splits["train"]
rng = np.random.default_rng(0)
picks = list(rng.choice(np.flatnonzero(labels == 0), 4, replace=False)) +         list(rng.choice(np.flatnonzero(labels == 1), 4, replace=False))
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, i in zip(axes.ravel(), picks):
    ax.imshow(load_image(paths[i]))
    ax.set_title(CLASSES[labels[i]], fontweight="bold")
    ax.axis("off")
fig.suptitle("Training images (resized to 224x224)", fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/visualizations/sample_images.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Smoke test, then train

`main.py --mode train` trains, picks the epoch with the best validation AUC, then evaluates on the test set and writes the Grad-CAM figures.

In [ ]:
!python main.py

In [ ]:
!python main.py --mode train

## 5. Results

In [ ]:
import json
from IPython.display import Image, display

m = json.load(open("outputs/reports/metrics.json"))
print(json.dumps({k: round(v, 4) for k, v in m["metrics"].items()}, indent=2))
print("Confusion matrix:", m["confusion_matrix"])
print("Best epoch:", m.get("best_epoch"), "of", m.get("epochs_trained"))

for name in ["training_curves", "confusion_matrix", "roc_curve", "gradcam_example",
             "gradcam_examples", "prediction_examples"]:
    display(Image(f"outputs/visualizations/{name}.png", width=900))

## 6. Download the results

Send `results.zip` and this notebook (*File → Download → Download .ipynb*) back so they can be committed to the repo.

In [ ]:
!rm -f results.zip
!zip -qr results.zip outputs/visualizations outputs/reports outputs/logs/history.csv outputs/logs/history.json
!ls -lh results.zip
from google.colab import files
files.download("results.zip")

In [ ]:
# Optional: the trained model (about 100 MB with optimizer state)
# files.download("outputs/models/best_model.keras")